In [1]:
from nltk.corpus import movie_reviews

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import numpy as np
from collections import Counter
import re

In [3]:
class SimpleTokenizer:
    def __init__(self, num_words=10000 , oov_token='UNK'):
        self.num_words = num_words
        self.word_index = {}
        self.index_word = {}
        self.oov_token = oov_token

    def clean_text(self, text):
        return re.findall(r'\w+', text.lower())
    
    def fit_on_texts(self, texts):
        word_counts = Counter()
        for text in texts:
            word_counts.update(self.clean_text(text))
        
        most_common = word_counts.most_common(self.num_words - 2)
        # 0 : padding , 1 : oov_token
        self.word_index = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
        self.word_index[self.oov_token] = 1
        self.index_word = {idx: word for word, idx in self.word_index.items()}

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            cleaned_text = self.clean_text(text)
            sequence = [self.word_index.get(word, self.word_index[self.oov_token]) for word in cleaned_text]
            sequences.append(sequence)
        return sequences

def pad_sequences(sequences, maxlen , padding='pre' , truncating='pre'):
    for seq in sequences:
        if len(seq) > maxlen:
            if truncating == 'pre':
                del seq[:-maxlen]
            else:
                del seq[maxlen:]
        else:
            if padding == 'pre':
                seq[:0] = [0] * (maxlen - len(seq))
            else:
                seq.extend([0] * (maxlen - len(seq)))
    return sequences

In [4]:
text = "This is a sample text, with punctuation!"
sim = SimpleTokenizer(num_words=10000 , oov_token='UNK')
sim.fit_on_texts([text])
print(sim.texts_to_sequences([text]))

[[2, 3, 4, 5, 6, 7, 8]]


In [5]:
reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]
text = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(text)
requens = sim.texts_to_sequences(text)

In [6]:
features = pad_sequences(requens , 500)
features

[[6,
  3,
  10,
  88,
  59,
  216,
  100,
  21,
  217,
  36,
  18,
  54,
  218,
  7,
  219,
  5,
  30,
  220,
  100,
  4,
  51,
  16,
  101,
  22,
  3,
  221,
  10,
  222,
  223,
  2,
  102,
  224,
  37,
  4,
  37,
  103,
  21,
  42,
  225,
  6,
  226,
  104,
  105,
  5,
  227,
  44,
  10,
  13,
  30,
  8,
  106,
  87,
  3,
  8,
  228,
  107,
  13,
  108,
  109,
  7,
  110,
  16,
  3,
  229,
  7,
  230,
  7,
  110,
  3,
  231,
  232,
  47,
  233,
  234,
  111,
  4,
  60,
  19,
  38,
  99,
  112,
  235,
  113,
  45,
  236,
  12,
  2,
  237,
  59,
  50,
  2,
  238,
  114,
  10,
  9,
  2,
  239,
  4,
  21,
  240,
  241,
  20,
  242,
  29,
  13,
  46,
  23,
  115,
  243,
  49,
  6,
  3,
  33,
  116,
  2,
  244,
  61,
  245,
  46,
  22,
  6,
  2,
  62,
  105,
  9,
  117,
  118,
  7,
  38,
  5,
  63,
  246,
  6,
  119,
  16,
  3,
  64,
  84,
  18,
  2,
  38,
  2,
  30,
  22,
  9,
  65,
  112,
  21,
  120,
  2,
  247,
  248,
  26,
  249,
  29,
  13,
  10,
  9,
  34,
  250,
  251,
  38,
  252,

In [7]:
# RNN 모델
class RNNmodel(nn.Module):
    def __init__(self,vocab_size , embedding_dim , hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size , embedding_dim)
        self.rnn = nn.RNN(embedding_dim ,hidden_dim , batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim , 32),
            nn.ReLU(),
            nn.Linear(32,1))
    def forward(self,x):
        x = self.embedding(x)
        _,hn = self.rnn(x)   # output , hn
        # output : 모든 시점 (time - step) 의 숨겨진 상태 : 각 단어 (시점) 을 거칠때마다 계산된 모든 hidden state 를 모아놓음
        # seq2seq 모델은 각 단어마다 결과를 내야하는 개체면 인식(NER)
        # hn : 마지막 시점 상태 : 전체 문장을 다 읽고 최종적으로 요약한 정보 : 문서분류
        return self.fc(hn.squeeze(0))


In [8]:
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts)
requens = sim.texts_to_sequences(texts)
features = pad_sequences(requens,500)
features = torch.LongTensor(features)
print(features.shape)

features = nn.Embedding(500,32)(features)
outputs , hn = nn.RNN(32,64,batch_first=True)(features)
outputs.shape

torch.Size([2, 500])


torch.Size([2, 500, 64])

In [ ]:
# 데이터를 가져오기
# x y 분할
# 토크나이저 + pad_seq --> 문자를 숫자로 변환
# train_test_split
# Torch tensor 변환
# TensorDataset -> Dataloader
# 모델 생성
# 옵티마이저 정의
# 손실함수 정의